In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

model = ChatOllama(model="llama3.2", temperature=0.3)
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='c840d733-58f8-4d08-921b-cf296a1bc07d'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T04:44:01.25691874Z', 'done': True, 'done_reason': 'stop', 'total_duration': 795503563, 'load_duration': 152138646, 'prompt_eval_count': 301, 'prompt_eval_duration': 389500719, 'eval_count': 23, 'eval_duration': 241229808, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09e3a-ac49-7152-a1be-5c52fd4930a5-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'fb617b9c-a587-4b8a-b5e9-1da5b83b8cc7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 301, 'output_tokens': 23, 'total_tokens': 324}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "langchain-mcp-ad

## Online MCP

In [8]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [9]:
agent = create_agent(
    model="ollama:llama3.2",
    tools=tools,
)

In [10]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='5e7af347-ba3a-4830-82d6-d5e4841c2f13'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-14T04:44:11.305255563Z', 'done': True, 'done_reason': 'stop', 'total_duration': 658964915, 'load_duration': 141599978, 'prompt_eval_count': 344, 'prompt_eval_duration': 270694088, 'eval_count': 21, 'eval_duration': 235689073, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a09e3a-d412-78d1-96d3-3a9bb9895e06-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'c8c703da-3e34-46e2-b5f3-2a5db4978ce0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 344, 'output_tokens': 21, 'total_tokens': 365}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-09-14T00: